[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhihsheng/eeqt30001/blob/main/assignment/starter.ipynb)　←　點此在 Google Colab 直接開啟本 notebook（免安裝）

# EEQT30001 量子科技與工程｜單元二 QuTiP 加分作業

**姓名**：＿＿＿＿＿＿　**學號**：＿＿＿＿＿＿

| 項目 | 說明 |
|------|------|
| 發布 | 第 7 週課後（2026-10-20） |
| 繳交 | 第 11 週期中考二當日（2026-11-17），notebook 檔上傳 E3，檔名 `學號_qutip.ipynb` |
| 加分 | Q1 +2、Q2 +2、Q3 +1，**至多 +5 分**加於期中考二卷面（加總以 100 分為上限） |
| 規則 | 不強制、不催交、不受理遲交——做的人加分，不做的人無事 |
| AI 工具 | **明文允許**（ChatGPT／Claude／Copilot⋯）——評分只看結果與標註，不查核過程 |
| 評分 | **完成導向**：程式可執行、圖表有標註即給分 |

建議環境：**Google Colab**（免安裝）。本機亦可，需 Python ≥ 3.10。

## 0｜環境安裝

第一次執行先跑下面這格（Colab 約需 1 分鐘）；裝完若 import 失敗，重啟 runtime 再跑一次。

In [ ]:
%pip install --quiet qutip

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import (basis, coherent, thermal_dm, squeeze, destroy, num,
                   qeye, tensor, expect, wigner)
import qutip

print("QuTiP version:", qutip.__version__)
N = 30  # Fock-space truncation (single mode)

## Q1（+2）三種量子光態的 Wigner 函數

畫出下列三個態的 Wigner 函數 $W(x,p)$（三張並排的 2D 圖），並在圖上**標註各自的特徵**：

1. **單光子 Fock 態** $|1\rangle$ —— 中心的**負值區**（非古典性的直接證據）
2. **相干態** $|\alpha=2\rangle$ —— 從原點**位移**的高斯圓斑（雷射＝被位移的真空）
3. **壓縮真空** $S(r{=}1)|0\rangle$ —— **橢圓**：一個象限變窄（$e^{-r}$）、共軛象限變寬（$e^{+r}$）

提示：
- `wigner(psi, xvec, xvec)`，`xvec = np.linspace(-5, 5, 201)`
- QuTiP 預設約定即課堂的 $X=(a+a^\dagger)/\sqrt2$：相干態中心在 $x=\sqrt2\,\alpha$
- 配色建議 `cmap="RdBu_r"` 並讓色階以 0 為中心（負值一眼可見）；標註用 `ax.annotate()` 或 `ax.text()`

In [ ]:
xvec = np.linspace(-5, 5, 201)
states = {"Fock |1>": basis(N, 1),
          "Coherent |a=2>": coherent(N, 2.0),
          "Squeezed vac S(1)|0>": squeeze(N, 1.0) * basis(N, 0)}

# --- 示範：真空態的 Wigner 函數（先跑跑看，再仿照畫出上面三個態）---
W0 = wigner(basis(N, 0), xvec, xvec)
fig, ax = plt.subplots(figsize=(4, 3.4))
cs = ax.contourf(xvec, xvec, W0, 100, cmap="RdBu_r",
                 vmin=-abs(W0).max(), vmax=abs(W0).max())
ax.set_xlabel("x"); ax.set_ylabel("p"); ax.set_title("Vacuum |0>")
ax.set_aspect("equal"); fig.colorbar(cs, ax=ax)
plt.show()

# TODO: 1x3 subplots，對 states 迴圈畫出三個態的 Wigner 函數
# TODO: 每張圖標註特徵（負值區／位移量 sqrt(2)*alpha／壓縮橢圓軸）

## Q2（+2）$g^{(2)}(0)$：一個數字分出三種光

計算下列三種光的二階關聯函數，驗證教科書值 **2／1／0**：

| 光源 | 態 | 理論值 |
|------|----|--------|
| 熱光（chaotic） | `thermal_dm(N, 2.0)`（$\bar n=2$） | 2 |
| 相干光（雷射） | $\vert\alpha=2\rangle$ | 1 |
| 單光子 | $\vert 1\rangle$ | 0 |

$$g^{(2)}(0)=\frac{\langle a^\dagger a^\dagger a a\rangle}{\langle a^\dagger a\rangle^2}$$

提示：`expect()` 同時接受純態（ket）與密度矩陣；輸出請做成**表格或長條圖並標註**理論值 2、1、0。

In [ ]:
a = destroy(N)

# --- 課堂示範（第 7 週投影片同款）：先跑跑看 ---
demo = {"Fock |1>": basis(N, 1), "Coherent": coherent(N, 2.0),
        "Squeezed": squeeze(N, 1.0) * basis(N, 0)}
for name, psi in demo.items():
    g2 = expect(a.dag() * a.dag() * a * a, psi) / expect(a.dag() * a, psi) ** 2
    print(name, "g2(0) =", round(g2, 3))

# TODO: 換成作業要求的三種光——thermal_dm(N, 2.0)、coherent(N, 2.0)、basis(N, 1)
# TODO: 印出表格或畫長條圖，標註理論值 2 / 1 / 0

## Q3（+1）N00N 態的量子 Fisher 資訊：$F_Q=N^2$

雙模 N00N 態
$$|\mathrm{N00N}\rangle=\frac{|N,0\rangle+|0,N\rangle}{\sqrt2},\qquad
G=\hat n_a\ (\text{相位臂光子數}),\qquad F_Q=4(\Delta G)^2$$

數值驗證 $N=2,4,6$ 時 $F_Q=N^2$（Heisenberg 極限）。
對照：相干態 $F_Q=4\langle n\rangle$（標準量子極限 SQL）——糾纏把 Fisher 資訊由 $\propto N$ 抬到 $N^2$。

提示：
- 兩模空間用 `tensor()`：$|N,0\rangle$ = `tensor(basis(d, N_ph), basis(d, 0))`，`d = N_ph + 1`
- 疊加後用 `.unit()` 歸一化；$\hat n_a$ = `tensor(num(d), qeye(d))`
- $(\Delta G)^2=\langle G^2\rangle-\langle G\rangle^2$

In [ ]:
def noon_state(N_ph):
    """(|N,0> + |0,N>)/sqrt(2) in a two-mode Fock space."""
    d = N_ph + 1
    # TODO: psi = tensor(basis(d, N_ph), basis(d, 0)) + tensor(..., ...)
    # TODO: return psi.unit()
    pass

# TODO: for N_ph in [2, 4, 6]:
#           psi = noon_state(N_ph)
#           n_a = tensor(num(N_ph + 1), qeye(N_ph + 1))
#           FQ  = 4 * (expect(n_a ** 2, psi) - expect(n_a, psi) ** 2)
#           print N_ph, FQ, 理論值 N_ph**2

## 繳交前 checklist

- [ ] `Runtime → Run all`（或 `Kernel → Restart & Run All`）全部 cell 無錯誤
- [ ] 三題的圖表**都有標註**（特徵、理論值、軸標籤）
- [ ] 最上方姓名學號已填
- [ ] 檔名改為 `學號_qutip.ipynb`，上傳 E3

有問題可用任何 AI 工具，或於課後提問。祝順利！